In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

settlements_saws = pd.read_csv("settlements_saws.csv", low_memory = False)
eskom = pd.read_csv("eskom_clean.csv", parse_dates = ["date_time"])

print(f"Settlements_saws: {settlements_saws.shape}")
print(f"Eskom: {eskom.shape}")

# consistent plot style for all visualisations
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Demand Distribution Across Provinces

In [ ]:
fig, ax = plt.subplots(figsize = (10, 6))

province_demand = (
    settlements_saws.groupby("province")["demand"].sum().sort_values(ascending = True)
)

province_demand.plot(kind = "barh", ax = ax, color = "#378ADD", edgecolor = "none")

ax.set_title("Total Electricity Demand by Province", fontsize = 14, fontweight = "bold", pad = 15)
ax.set_xlabel("Total Demand (MW)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig("vis1_demand_by_province.png", bbox_inches = "tight")
plt.show()

## Solar Potential Map

In [ ]:
import folium
from folium.plugins import HeatMap
import branca.colormap as cm

# create base map centered on SA 
sa_map = folium.Map(
    location = [-28.5, 24.7],
    zoom_start = 6,
    tiles = "CartoDB positron" # clean light background
)

# build colour scale based on pv_value range
colormap = cm.LinearColormap(
    colors = ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"],
    vmin = settlements_saws["pv_value"].min(),
    vmax = settlements_saws["pv_value"].max(),
    caption = "Solor Potential (pv_value - kWh/m²/year)"
)

# sample random 20 000 villages to keep map responsive
sample = settlements_saws.sample(n = 20000, random_state = 42)

# plot each village as circle coloured by pv_value
for _, row in sample.iterrows():
    folium.CircleMarker(
        location = [row["lat"], row["lon"]],
        radius = 2,
        color = colormap(row["pv_value"]),
        fill = True,
        fill_opacity = 0.6,
        popup = f"{row["village_name"]}<br>PV: {row["pv_value"]:.0f} kWh/m²"
    ).add_to(sa_map)
    
# add colour scale legend
colormap.add_to(sa_map)

sa_map.save("vis2_solar_potential_map.html")
print("Saved: vis2_solar_potential_map.html")
print("Open this file in your browser to interactively explore the map")

## Average Cloud Cover by Station

In [ ]:
fig, ax = plt.subplots(figsize = (9, 5))

cloud_data = (
    settlements_saws.groupby("nearest_station_name")["avg_cloud_octas"].first().dropna().sort_values(ascending = True)
)

bars = cloud_data.plot(kind = "barh", ax = ax, color = "#7F77DD", edgecolor = "none")

# add reference line at 50% cloud cover
for i, v in enumerate(cloud_data.values):
    ax.text(v + 0.05, i, f"{v:.2f}", va = "center", fontsize = 10)

ax.set_title("Average Cloud Cover by SAWS Station (2021-2026)", fontsize = 14, fontweight = "bold", pad = 15)
ax.set_xlabel("Average Cloud Cover (octas, 0 = clear, 8 = overcast)")
ax.set_ylabel("")
ax.set_xlim(0, 8)
ax.legend(fontsize = 9)

plt.tight_layout()
plt.savefig("vis3_cloud_cover_by_station.png", bbox_inches = "tight")
plt.show()

## Temperature Trends Over Time from SAWS

In [ ]:
import matplotlib.dates as mdates

saws = pd.read_csv("saws_clean.csv", parse_dates = ["date"])

fig, ax = plt.subplots(figsize = (14, 6))
colors = plt.cm.tab10.colors

for idx, (station, grp) in enumerate(saws.groupby("station_name")):
    monthly = (
        grp.groupby(grp["date"].dt.to_period("M"))["max_temp_c"].mean()
    )
    monthly.index = monthly.index.to_timestamp()
    ax.plot(monthly.index, monthly.values, label = station, color = colors[idx % 10], lw = 1.5, alpha = 0.85)

ax.set_title("Monthly Average Maximum Temperature by Station (2021-2026)", fontsize = 14, fontweight = "bold", pad = 15)
ax.set_ylabel("Average Max Temperature (°C)")
ax.set_xlabel("")
ax.legend(fontsize = 8, loc = "upper right")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig("vis4_temp_trends.png", bbox_inches = "tight")
plt.show()

## Demand vs Solar Potential

In [ ]:
fig, ax = plt.subplots(figsize = (10, 6))

sample = settlements_saws.sample(n = 15000, random_state = 42)

scatter = ax.scatter(
    sample["pv_value"],
    sample["demand"],
    c = sample["security_risk_score"],
    cmap = "RdYlGn_r",
    alpha = 0.4,
    s = 8,
    edgecolors = "none"
)

z = np.polyfit(sample["pv_value"], sample["demand"], 1)
p = np.poly1d(z)
pv_range = np.linspace(sample["pv_value"].min(), sample["pv_value"].max(), 100)
ax.plot(pv_range, p(pv_range), color = "black", lw = 1.5, linestyle = "--", label = "Trend line")

# colour bar for security risk
cbar = plt.colorbar(scatter, ax = ax)
cbar.set_label("Security Risk (0 = low, 1 = medium, 2 = high)", fontsize = 9)
cbar.set_ticks([0, 1, 2])

ax.set_title("Village Demand vs Solar Potential\ncoloured by Security Risk", fontsize = 14, fontweight = "bold", pad = 15)
ax.set_xlabel("Solar Potential (pv_value - kWh/m²/year)")
ax.set_ylabel("Electricty Demand (MW)")
ax.legend(fontsize = 9)

# add correlation value 
corr = settlements_saws["pv_value"].corr(settlements_saws["demand"])
ax.text(0.05, 0.95, f"Correlation: {corr:.3f}", transform = ax.transAxes, fontsize = 10, verticalalignment = "top",
        bbox = dict(boxstyle = "round", facecolor = "white", alpha = 0.8))

plt.tight_layout()
plt.savefig("vis5_demand_vs_solar_potential.png", bbox_inches = "tight")
plt.show()

## Security Risk Distribution

In [ ]:
fig, ax = plt.subplots(figsize = (12, 6))

risk_province = (
    settlements_saws.groupby(["province", "security_risk_score"]).size().unstack(fill_value = 0).rename(columns = {0: "Low", 1: "Medium", 2: "High"})
)

# convert to percentages
risk_province_pct = risk_province.div(risk_province.sum(axis = 1), axis = 0) * 100

risk_province_pct.plot(
    kind = "barh", stacked = True, ax = ax, color = ["#1D9E75", "#F4A623", "#D85A30"], edgecolor = "none"
)

ax.set_title("Security Risk Distribution by Province", fontsize = 14, fontweight = "bold", pad = 15)
ax.set_xlabel("Percentage of Viallages (%)")
ax.set_ylabel("")
ax.legend(title = "Security Risk", bbox_to_anchor = (1.01, 1), loc = "upper left")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

plt.tight_layout()
plt.savefig("vis6_security_risk_by_province.png", bbox_inches = "tight")
plt.show()

## Eskom Renewable Energy Gap Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize = (14, 10))

# monthly avg demand vs RE output plot
eskom["month_period"] = eskom["date_time"].dt.to_period("M")

monthly = eskom.groupby("month_period").agg(
    avg_demand = ("residual_demand", "mean"),
    avg_re_output = ("total_re", "mean"),
    avg_re_gap = ("re_gap_mw", "mean") if "re_gap_mw" in eskom.columns else ("total_re", "mean")
).reset_index()
monthly["month_period"] = monthly["month_period"].dt.to_timestamp()

axes[0].plot(monthly["month_period"], monthly["avg_demand"], color = "#D85A30", lw = 1.5, label = "Total Demand (MW)")
axes[0].plot(monthly["month_period"], monthly["avg_re_output"], color = "#1D9E75", lw = 1.5, label = "RE Output (MW)")
axes[0].fill_between(monthly["month_period"],
                     monthly["avg_re_output"],
                     monthly["avg_demand"], alpha = 0.15, color = "#D85A30", label = "RE Gap")
axes[0].set_title("Monthly Average Demand vs Renewable Energy Output (2021-2026)", fontsize = 13, fontweight = "bold", pad = 15)
axes[0].set_ylabel("Power (MW)")
axes[0].legend(fontsize = 9)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[0].xaxis.set_major_locator(mdates.YearLocator())

# RE penetration % over time plot
eskom["re_penetration_pct"] = (eskom["total_re"] / eskom["residual_demand"]) * 100
monthly_pct = eskom.groupby("month_period")["re_penetration_pct"].mean().reset_index()
monthly_pct["month_period"] = monthly_pct["month_period"].dt.to_timestamp()

axes[1].plot(monthly_pct["month_period"], monthly_pct["re_penetration_pct"], color = "#7F77DD", lw = 1.5)
axes[1].fill_between(monthly_pct["month_period"],
                     monthly_pct["re_penetration_pct"], alpha = 0.2, color = "#7F77DD")
axes[1].axhline(y = monthly_pct["re_penetration_pct"].mean(), color = "black", linestyle = "--", lw = 1, label = f"Average: {monthly_pct["re_penetration_pct"].mean():.1f}%")
axes[1].set_title("Renewable Energy Penetration Rate Over Time (%)", fontsize = 13, fontweight = "bold", pad = 15)
axes[1].set_ylabel("RE as % of Total Demand")
axes[1].set_xlabel("")
axes[1].legend(fontsize = 9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[1].xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig("vis7_eskom_re_gap.png", bbox_inches = "tight")
plt.show()